# Testing for Conductivity

Reusable function definitions only. Original project inputs, run settings and saved outputs are not included. Running these cells defines functions; it does not read a dataset. Start with the [synthetic walkthrough](../examples/synthetic_walkthrough.ipynb) for a complete small run.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "helper" / "pipeline_utils.py").is_file()
             and (p / "examples" / "synthetic_demo.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the notebook inside this repository.")
for folder in (ROOT, ROOT / "Invert-PropOnly-saveCSV", ROOT / "Invert-Prop+cond-saveCSV"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))


In [ ]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import r2_score

import os
from helper.pipeline_utils import Net



def infer_conductivity(
    in_dim: int,
    out_dim: int,
    hidden_dims: list,
    activation: str,
    x_scaler_path: str,
    y_scaler_path: str,
    test_path: str,
    model_path: str
) -> float:
    """
    Load scalers and model, run predictions on `test_path`, compute and
    print R² for the second output, then plot error histograms and
    Actual-vs-Predicted scatter plots for all outputs.

    Args:
        in_dim (int): Number of input features.
        out_dim (int): Number of output targets.
        hidden_dims (list[int]): Sizes of hidden layers.
        activation (str): Activation name ("relu", "sigmoid", "softplus").
        x_scaler_path (str): Path to saved StandardScaler for X.
        y_scaler_path (str): Path to saved StandardScaler for Y.
        test_path (str): Path to CSV containing test data.
        model_path (str): Path to saved PyTorch model (.pth).

    Returns:
        float: R² score for the second output dimension.
    """
    # [1] Load scalers
    with open(x_scaler_path, "rb") as f:
        x_scaler = pickle.load(f)
    with open(y_scaler_path, "rb") as f:
        y_scaler = pickle.load(f)

    # [2] Read test CSV and extract column names for outputs
    df_test = pd.read_csv(test_path)
    output_cols = df_test.columns[in_dim:].tolist()

    # [3] Scale inputs & targets
    X_test = x_scaler.transform(df_test.iloc[:, :in_dim].values)
    Y_test = y_scaler.transform(df_test.iloc[:, in_dim:].values)
    X_tensor = torch.from_numpy(X_test).float()

    # [4] Initialize & load the model
    model = Net(
        in_dim=in_dim,
        hidden_dims=hidden_dims,
        out_dim=out_dim,
        activation_str=activation
    )
    model.load_state_dict(torch.load(model_path, map_location="cpu"))
    model.eval()

    # [5] Run inference
    with torch.no_grad():
        Y_pred_scaled = model(X_tensor).numpy()

    # [6] Inverse-transform to original scale
    Y_pred = y_scaler.inverse_transform(Y_pred_scaled)
    Y_true = y_scaler.inverse_transform(Y_test)

    # [7] Compute and print R² for the second output
    r2_second = r2_score(Y_true[:, 1], Y_pred[:, 1])
    print(f"R² score for second output ({output_cols[1]}): {r2_second:.4f}")

    # [8] Plot histograms of prediction errors for each output
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for i, ax in enumerate(axes.flat):
        errors = Y_pred[:, i] - Y_true[:, i]
        ax.hist(errors, bins=30, edgecolor='k', alpha=0.7)
        ax.set_title(f"Output {i+1} ({output_cols[i]}): Pred − Actual")
        ax.set_xlabel("Error")
        ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()

    # [9] Plot Actual vs. Predicted scatter plots for each output
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for i, ax in enumerate(axes.flat):
        actual = Y_true[:, i]
        pred   = Y_pred[:, i]
        mn, mx = min(actual.min(), pred.min()), max(actual.max(), pred.max())

        ax.scatter(actual, pred, s=10, alpha=0.6)
        ax.plot([mn, mx], [mn, mx], 'r--', linewidth=1)
        ax.set_title(f"Output {i+1} ({output_cols[i]}): Actual vs Pred")
        ax.set_xlabel("Actual")
        ax.set_ylabel("Predicted")
        ax.set_xlim(mn, mx)
        ax.set_ylim(mn, mx)
    plt.tight_layout()
    plt.show()

    return r2_second


